# Evaluation of Structure-Aware Statistical Reachability Analysis of Program States

## Folder structure expected
```
Data/Blackbox/
  parsera/
    run_01/
      ft_input_cov.bin
      ft_input_cov.idx
    run_02/ ...
  parserb/ ...
  ...
```

**`GENERATE_PKL_FROM_BINARY = True`** → decode `.bin`/`.idx` files and write pkl files to `PKL_DIR`.  
**`GENERATE_PKL_FROM_BINARY = False`** → skip generation; load existing pkl files from `PKL_DIR` directly.

---
## Section 1 — Imports & Configuration

In [ ]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib

In [ ]:
import pickle
import struct
import lzma
import zlib
import json
import math
from array import array
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────────────
BINARY_DIR  = Path("Data")                # root of binary execution stores
PKL_DIR     = Path("Data")           # pkl files (generated or pre-existing)
RESULTS_DIR = Path("results")        # output CSV results
ICFG_DIR    = Path("icfg")               # per-parser iCFG JSON: icfg/parsera.json
GT_DIR      = Path("ground_truth_structural_logprob")  # per-block ln(Pr(b)): ground_truth_logprob/parsera.json

PKL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ── PKL generation flag ────────────────────────────────────────────────────
# True  → decode binary stores (.bin + .idx) and write pkl files to PKL_DIR.
#          Existing pkl files are overwritten.
# False → skip generation entirely; load existing pkl files from PKL_DIR.
GENERATE_PKL_FROM_BINARY = False

# ── Campaign checkpoints (fraction of total executions) ────────────────────
CHECKPOINTS = [0.10, 0.25, 0.50, 0.75, 1.00]

# ── Decision-depth bins ────────────────────────────────────────────────────
DEPTH_BINS = {"shallow": (0, 10), "medium": (11, 40), "deep": (41, float("inf"))}

# ── Zero-coincidence threshold ─────────────────────────────────────────────
ZC_THRESHOLD = 1e-6

print("Configuration loaded.")
print(f"  BINARY_DIR              : {BINARY_DIR.resolve()}")
print(f"  PKL_DIR                 : {PKL_DIR.resolve()}")
print(f"  RESULTS_DIR             : {RESULTS_DIR.resolve()}")
print(f"  ICFG_DIR                : {ICFG_DIR.resolve()}")
print(f"  GT_DIR                  : {GT_DIR.resolve()}")
print(f"  CHECKPOINTS             : {CHECKPOINTS}")
print(f"  GENERATE_PKL_FROM_BINARY: {GENERATE_PKL_FROM_BINARY}")


Configuration loaded.
  BINARY_DIR              : /Users/natt0753/USYD/Papers_Submitted/ICSE27/Analysis/icse2027/Data
  PKL_DIR                 : /Users/natt0753/USYD/Papers_Submitted/ICSE27/Analysis/icse2027/Data
  RESULTS_DIR             : /Users/natt0753/USYD/Papers_Submitted/ICSE27/Analysis/icse2027/results
  ICFG_DIR                : /Users/natt0753/USYD/Papers_Submitted/ICSE27/Analysis/icse2027/icfg
  GT_DIR                  : /Users/natt0753/USYD/Papers_Submitted/ICSE27/Analysis/icse2027/ground_truth_structural_logprob
  EVAL_FRACTION           : 0.5
  CHECKPOINTS             : [0.1, 0.25, 0.5, 0.75, 1.0]
  GENERATE_PKL_FROM_BINARY: False


---
## Section 2 — PKL Generation

When `GENERATE_PKL_FROM_BINARY = True`, reads each binary execution store
(`ft_input_cov.bin` + `ft_input_cov.idx`) and saves a CSR-format incidence
matrix pkl.

When `GENERATE_PKL_FROM_BINARY = False`, this section only defines helper
functions — no files are written.

**Matrix shape:** `(n_executions × total_blocks)` — each row = one execution,
each column = one basic block, value = 0 or 1 (binary hit/miss).

In [ ]:
# ── Varint decoder ──────────────────────
def _decode_varint(data: bytes, pos: int):
    result = shift = 0
    while True:
        byte = data[pos]; pos += 1
        result |= (byte & 0x7F) << shift
        if not (byte & 0x80):
            break
        shift += 7
    return result, pos


# ── Binary store iterator ──────────────────────────────────────────────────
def iter_binary_store(bin_path: Path, idx_path: Path):
    """
    File format (trace_store.py):
      .idx — fixed 8-byte little-endian uint64 offsets, one per record
      .bin — records of the form:
                [varint: payload_byte_len]
                [varint: n_blocks]
                [n_blocks * varint: delta-encoded sorted block IDs]
    """
    # Load offset index
    offsets = []
    with idx_path.open("rb") as f:
        while True:
            raw = f.read(8)
            if len(raw) < 8:
                break
            (off,) = struct.unpack("<Q", raw)
            offsets.append(off)

    # Stream bin file
    with bin_path.open("rb") as f:
        data = f.read()

    for off in offsets:
        _payload_len, pos = _decode_varint(data, off)  
        n_blocks, pos     = _decode_varint(data, pos)
        blocks: list[int] = []
        prev = 0
        for _ in range(n_blocks):
            delta, pos = _decode_varint(data, pos)
            val = prev + delta
            blocks.append(val)
            prev = val
        yield blocks


# ── PKL writer ─────────────────────────────────────────────────────────────
def generate_pkl_from_binary(bin_path: Path, idx_path: Path,
                              pkl_path: Path, total_blocks: int) -> dict:
    """Decode one binary store and write a CSR incidence-matrix pkl.
    """
    pkl_path.parent.mkdir(parents=True, exist_ok=True)

    row_ptr      = array("I", [0])   
    col_idx      = array("I")       
    n_executions = 0

    for blocks in iter_binary_store(bin_path, idx_path):
        unique_blocks = sorted(set(blocks))
        col_idx.extend(unique_blocks)
        row_ptr.append(len(col_idx))
        n_executions += 1

    matrix = {
        "matrix_format": "csr_binary_incidence_v1",
        "shape":          (n_executions, total_blocks),
        "row_ptr":        row_ptr,
        "col_idx":        col_idx,
        "data_value":     1,
        "block_ids":      list(range(total_blocks)),
    }

    with pkl_path.open("wb") as f:
        pickle.dump(matrix, f, protocol=pickle.HIGHEST_PROTOCOL)

    size_mb = pkl_path.stat().st_size / (1024 * 1024)
    return {
        "status":          "generated",
        "shape":           (n_executions, total_blocks),
        "nonzero_entries": len(col_idx),
        "file_size_mb":    round(size_mb, 2),
    }


# ── PKL loader ─────────────────────────────────────────────────────────────
def load_pkl(pkl_path: Path) -> dict:
    with pkl_path.open("rb") as f:
        m = pickle.load(f)

    fmt = m.get("matrix_format", "")

    if fmt == "csr_binary_incidence_v1":
        return m

    if fmt == "csr_binary_incidence_v3_compressed":
        compression = m["compression"]
        def _decompress(data):
            if compression == "lzma": return lzma.decompress(data)
            if compression == "zlib": return zlib.decompress(data)
            return data
        row_ptr = array("I", _decompress(m["row_ptr_bytes"]))
        col_idx = array("H", _decompress(m["col_idx_bytes"]))
        unique_exec_ids = m["unique_exec_ids"]
        execution_ids   = [unique_exec_ids[i] for i in m["exec_id_index"]]
        return {
            "matrix_format": "csr_binary_incidence_v1",
            "shape":         m["shape"],
            "row_ptr":       row_ptr,
            "col_idx":       col_idx,
            "data_value":    m["data_value"],
            "block_ids":     m["block_ids"],
            "execution_ids": execution_ids,
        }

    raise ValueError(f"Unknown matrix_format: {fmt!r}")


print("PKL generation functions defined.")

PKL generation functions defined.


<!-- ---
## Section 3 — Estimators

### GoTu (Good-Turing)
$$\hat{U}_G = \frac{|V_1|}{n}$$
where $V_1$ = blocks seen exactly once, $n$ = number of executions. -->


In [3]:
def compute_gotu(matrix: dict) -> float:
    """Good-Turing"""
    col_idx      = matrix["col_idx"]
    n            = matrix["shape"][0]
    block_counts = Counter(col_idx)
    v1           = sum(1 for c in block_counts.values() if c == 1)
    return v1 / n if n > 0 else 0.0



print("Estimator functions defined.")

Estimator functions defined.


---
## Section 3 — iCFG & Per-Block Ground Truth Loader

Loads per-parser iCFG (successor edges + decision depths) and exact per-block
reaching probabilities Pr(b) computed analytically from the grammar.

Expected files:
- `icfg/<parser>.json`         → `{"block_id": {"depth": int, "successors": [str, ...]}}`
- `ground_truth/<parser>.json` → `{"block_id": float}`  (exact Pr(b) values)

In [ ]:
def load_icfg(parser_name: str) -> dict:
    """Load iCFG for a parser.
    Returns dict: block_id (str) -> {"depth": int, "successors": [str, ...]}
    """
    path = ICFG_DIR / f"{parser_name}.json"
    with path.open() as f:
        return json.load(f)


def _logaddexp(a: float, b: float) -> float:
    """log(exp(a) + exp(b)) for natural-log values; handles -inf."""
    if a == float("-inf"):
        return b
    if b == float("-inf"):
        return a
    m = a if a > b else b
    return m + math.log(math.exp(a - m) + math.exp(b - m))


def compute_ground_truth_fixpoint(icfg: dict, tol: float = 1e-12,
                                  max_iter: int = 20000) -> dict:
    """
    Model (Lee & Bohme 2023, 1/k branch model)
    """
    from collections import deque
    NEG = float("-inf")
    preds = {b: [] for b in icfg}
    logw  = {}
    for b, info in icfg.items():
        k = len(info["successors"])
        logw[b] = (math.log(1.0 / k) if k >= 2 else 0.0)   # pass-through if k<=1
        for s in info["successors"]:
            if s in preds:
                preds[s].append(b)
    has_succ = {b for b, info in icfg.items() if info["successors"]}
    roots = sorted([b for b in icfg if b in has_succ and not preds[b]],
                   key=lambda x: int(x))
    if not roots:
        return {b: NEG for b in icfg}
    base = {b: NEG for b in icfg}
    log_root = -math.log(len(roots))
    for r in roots:
        base[r] = log_root
    # Gauss-Seidel update order: BFS from roots (fast convergence on DAG part)
    order, seen, q = [], set(roots), deque(roots)
    while q:
        u = q.popleft(); order.append(u)
        for s in icfg[u]["successors"]:
            if s in icfg and s not in seen:
                seen.add(s); q.append(s)
    for b in icfg:                      # append any nodes BFS could not reach
        if b not in seen:
            order.append(b)
    lp = dict(base)
    for _ in range(max_iter):
        max_delta = 0.0
        for b in order:
            acc = base[b]
            for p in preds[b]:
                pv = lp[p]
                if pv != NEG:
                    acc = _logaddexp(acc, pv + logw[p])
            if acc != lp[b]:
                if lp[b] == NEG:
                    max_delta = max(max_delta, 1.0)
                elif acc != NEG:
                    max_delta = max(max_delta, abs(acc - lp[b]))
                lp[b] = acc
        if max_delta < tol:
            break
    result = {}
    for b, info in icfg.items():
        if info["depth"] == 0 and not info["successors"]:
            result[b] = 0.0                       # non-grammar block, Pr = 1
        else:
            result[b] = min(0.0, lp.get(b, NEG))  # clamp tiny float overshoot
    return result


def load_ground_truth(parser_name: str) -> dict:
    """Per-block exact reaching probabilities ln Pr(b).
    """
    return compute_ground_truth_fixpoint(load_icfg(parser_name))


def get_depth_bin(depth: int) -> str:
    for bin_name, (lo, hi) in DEPTH_BINS.items():
        if lo <= depth <= hi:
            return bin_name
    return "deep"


print("iCFG loader + FIXPOINT ground-truth recompute + depth-bin functions defined.")

In [5]:
def build_prefix(matrix: dict, fraction: float):
    """Return (S_t, n_t, block_counts) for executions[:n_t].

    S_t          : set of block IDs reached in first n_t executions
    n_t          : number of executions used at this checkpoint
    block_counts : Counter of per-block hit counts up to n_t
    """
    col_idx = matrix["col_idx"]
    row_ptr = matrix["row_ptr"]
    n       = matrix["shape"][0]
    n_t     = max(1, int(n * fraction))

    S_t          = set()
    block_counts = Counter()
    for row in range(n_t):
        start, stop = row_ptr[row], row_ptr[row + 1]
        blocks = [col_idx[i] for i in range(start, stop)]
        S_t.update(blocks)
        block_counts.update(blocks)

    return S_t, n_t, block_counts


def compute_frontier(S_t: set, icfg: dict) -> set:
    """F_t = reached blocks with at least one unreached successor."""
    frontier = set()
    for b in S_t:
        b_str = str(b)
        if b_str in icfg:
            if any(int(s) not in S_t for s in icfg[b_str]["successors"]):
                frontier.add(b)
    return frontier


print("Checkpoint prefix and frontier functions defined.")

Checkpoint prefix and frontier functions defined.


---
## Section 4 — Structure-Aware Estimator (Hy-Path)


In [ ]:
ALPHA = 2                 # Laplace smoothing parameter (Lee & Bohme 2023)
MAX_HYPATH_DEPTH = 60     
MAX_HYPATH_PATHS = 20000  # safety cap on enumerated paths per block 


def _build_preds(icfg: dict) -> dict:
    """block_id(int-as-str key) -> list of predecessor block_ids (str)."""
    preds = {b: [] for b in icfg}
    for b, info in icfg.items():
        for s in info["successors"]:
            if s in preds:
                preds[s].append(b)
    return preds


def _enumerate_hy_paths(target, frontier, S_t, icfg,
                        max_depth=MAX_HYPATH_DEPTH, max_paths=MAX_HYPATH_PATHS):
    """FORWARD: frontier_node -> unreached nodes -> target."""
    n = 0
    for p1 in frontier:
        stack = [(p1, [], {p1})]
        while stack:
            node, path, visited = stack.pop()
            node_str = str(node)
            if node_str not in icfg:
                continue
            for succ_str in icfg[node_str]["successors"]:
                succ = int(succ_str)
                if succ in visited:
                    continue
                if succ == target:
                    yield (p1, path); n += 1
                    if n >= max_paths:
                        return
                elif succ not in S_t and len(path) < max_depth:
                    stack.append((succ, path + [succ], visited | {succ}))


def _enumerate_hy_paths_backward(target, frontier, S_t, icfg, preds,
                                 max_depth=MAX_HYPATH_DEPTH, max_paths=MAX_HYPATH_PATHS):
    """ BACKWARD recovery"""
    n = 0
    stack = [(target, [], {target})]
    while stack:
        cur, between, vis = stack.pop()
        for p in preds.get(str(cur), []):
            pi = int(p)
            if pi in vis:
                continue
            new_between = between if cur == target else [cur] + between
            if pi in frontier:
                yield (pi, new_between); n += 1
                if n >= max_paths:
                    return
            elif pi not in S_t and len(new_between) < max_depth:
                stack.append((pi, new_between, vis | {pi}))


def compute_hy_path_estimate(block, S_t, frontier, icfg, block_counts, n_t,
                             preds, max_depth=MAX_HYPATH_DEPTH):
    """Estimate Pr_struct(b) for a single unreached block b.
    """
    if n_t == 0:
        return 0.0, "zero_nt"

    def _accumulate(path_iter):
        tot = 0.0; found = False
        for p1, inter in path_iter:
            found = True
            c_p1   = block_counts.get(p1, 0)
            pr_hat = c_p1 / n_t
            first  = ALPHA / (c_p1 + 2 * ALPHA)
            path_pr = 1.0
            for nd in inter:
                nds = str(nd)
                if nds in icfg:
                    k = len(icfg[nds]["successors"])
                    if k > 0:
                        path_pr *= 1.0 / k
            tot += pr_hat * first * path_pr
        return tot, found

    # 1) forward search
    tot, found = _accumulate(
        _enumerate_hy_paths(block, frontier, S_t, icfg, max_depth))
    if found:
        return tot, "forward"

    # 2) backward-slice recovery
    tot, found = _accumulate(
        _enumerate_hy_paths_backward(block, frontier, S_t, icfg, preds, max_depth))
    if found:
        return tot, "backward"

    # 3) genuinely no structural path -> flagged global Laplace prior
    return ALPHA / (n_t + 2 * ALPHA), "isolated_prior"


def compute_all_hy_path_estimates(S_t, frontier, icfg, block_counts, n_t,
                                  all_blocks):
    """Run the hy-path estimator for all unreached blocks.
    Returns (estimates, methods): two dicts block_id(int) -> est / -> method.
    """
    preds = _build_preds(icfg)
    estimates, methods = {}, {}
    for b in (all_blocks - S_t):
        est, meth = compute_hy_path_estimate(
            b, S_t, frontier, icfg, block_counts, n_t, preds)
        estimates[b] = est
        methods[b]   = meth
    return estimates, methods


print("Hy-path estimator (forward + backward recovery + isolated flag) defined.")

---
## Section 5 — Per-Block Evaluation Metrics

Computes per-block log error and zero-coincidence violations against exact Pr(b),
then aggregates by decision-depth bin.

- **log-err(b, t)** = |log10(Pr_est(b)) − log10(Pr_true(b))|  
- **ZC violation**: estimator assigns near-zero but Pr(b) > ZC_THRESHOLD

In [ ]:
def compute_per_block_log_error(estimates: dict, ground_truth: dict,
                                 icfg: dict, methods: dict = None) -> list:
    """Compute log10 error for each unreached block with a valid ground truth.
    """
    methods = methods or {}
    rows = []
    for b, pr_est in estimates.items():
        b_str = str(b)
        if b_str not in ground_truth:
            continue
        pr_true = ground_truth[b_str]
        if math.isinf(pr_true):  # -inf means isolated/unreachable block — skip
            continue

        depth     = icfg[b_str]["depth"] if b_str in icfg else -1
        depth_bin = get_depth_bin(depth)

        log10_true   = pr_true / math.log(10)  # ln(Pr) -> log10(Pr)
        log_err      = abs(math.log10(pr_est) - log10_true) if pr_est > 0 else float("nan")
        zc_violation = (pr_est < ZC_THRESHOLD) and (pr_true > math.log(ZC_THRESHOLD))

        rows.append({
            "block_id":     b,
            "depth":        depth,
            "depth_bin":    depth_bin,
            "pr_true":      pr_true,
            "pr_est":       pr_est,
            "log_err":      log_err,
            "zc_violation": zc_violation,
            "est_method":   methods.get(b, "forward"),
        })
    return rows


def summarise_log_error_by_depth(per_block_rows: list,
                                 exclude_isolated: bool = True) -> dict:
    """Aggregate mean log_err and ZC rate per depth bin.
    """
    from collections import defaultdict
    bins = defaultdict(list)
    zc   = defaultdict(list)
    iso  = defaultdict(int)
    tot  = defaultdict(int)
    for r in per_block_rows:
        bn = r["depth_bin"]
        tot[bn] += 1
        is_iso = (r.get("est_method") == "isolated_prior")
        if is_iso:
            iso[bn] += 1
            if exclude_isolated:
                continue
        if not math.isnan(r["log_err"]):
            bins[bn].append(r["log_err"])
        zc[bn].append(float(r["zc_violation"]))
    result = {}
    for bin_name in ["shallow", "medium", "deep"]:
        v = bins[bin_name]; z = zc[bin_name]
        result[bin_name] = {
            "mean_log_err":  float(np.mean(v)) if v else float("nan"),
            "n_blocks":      len(v),
            "zc_rate":       float(np.mean(z)) if z else float("nan"),
            "n_isolated":    iso[bin_name],
            "isolated_rate": (iso[bin_name] / tot[bin_name]) if tot[bin_name] else float("nan"),
        }
    return result


print("Per-block metric functions defined (with isolated-prior flag + exclusion).")

---
## Section 6 — Helper Functions


In [10]:
def _mean(vals):
    v = [x for x in vals if not np.isnan(x)]
    return float(np.mean(v)) if v else float("nan")

def _std(vals):
    v = [x for x in vals if not np.isnan(x)]
    return float(np.std(v)) if v else float("nan")


print("Helper functions defined.")


Helper functions defined.


---
## Section 7 — Parser Metadata

`total_blocks` and `mrd` for each parser are defined directly in `PARSER_METADATA` below.

- `total_blocks` = total number of reachable basic blocks in the parser  
- `mrd` = Mean Reachability Depth computed from the parser CFG

**Edit `PARSER_METADATA` and run this cell before running Section 8.**

In [ ]:
# ── Parser metadata ────────────────────────────────────────────────────────
# Keys must match the parser folder names under BINARY_DIR (e.g. "parsera").
PARSER_METADATA = {
    "parsera": {"total_blocks": 16084, "mrd": 35.82},
    "parserb": {"total_blocks": 10484, "mrd": 24.81},
    "parserc": {"total_blocks": 17684, "mrd": 38.64},
    "parserd": {"total_blocks": 26684, "mrd": 53.75},
    "parsere": {"total_blocks": 9540, "mrd": 75.93},
}

print(f"PARSER_METADATA loaded for {len(PARSER_METADATA)} parsers.")
for name, meta in PARSER_METADATA.items():
    print(f"  {name}: total_blocks={meta['total_blocks']}  mrd={meta['mrd']}")

PARSER_METADATA loaded for 6 parsers.
  parsera: total_blocks=16084  mrd=35.82
  parserb: total_blocks=16084  mrd=35.82
  parserc: total_blocks=10484  mrd=24.81
  parserd: total_blocks=17684  mrd=38.64
  parsere: total_blocks=26684  mrd=53.75
  parserf: total_blocks=9540  mrd=75.93


---
## Section 8 — Run Full Pipeline

- If `GENERATE_PKL_FROM_BINARY = True`: decodes each `run_<N>/ft_input_cov.bin` + `.idx`,
  writes a pkl to `PKL_DIR/<parser>/<parser>_<run>.pkl`, then runs the analysis.
- If `GENERATE_PKL_FROM_BINARY = False`: loads existing pkls from `PKL_DIR` directly
  and runs the analysis — no binary files are read.

In [13]:
CKPT_CSV   = RESULTS_DIR / "structural/per_run_per_checkpoint.csv"
PB_CSV     = RESULTS_DIR / "structural/per_block_log_error.csv"
SUM_CSV    = RESULTS_DIR / "structural/summary.csv"

In [14]:
# ── Resume support: load already-completed keys from existing CSVs ─────────


def _load_done_keys(csv_path, key_cols):
    """Return a set of tuples for rows already written to csv_path."""
    if not csv_path.exists():
        return set()
    try:
        df = pd.read_csv(csv_path, usecols=key_cols)
        return set(zip(*[df[c] for c in key_cols]))
    except Exception:
        return set()

done_ckpt = _load_done_keys(CKPT_CSV, ["parser", "run", "checkpoint"])
done_pb   = _load_done_keys(PB_CSV,   ["parser", "run", "checkpoint"])
done_sum  = _load_done_keys(SUM_CSV,  ["parser"]) if SUM_CSV.exists() else set()

print(f"Resume state: {len(done_ckpt)} checkpoint rows already done.")
print(f"              {len(done_pb)}   per-block rows already done.")
print(f"              {len(done_sum)} parser summaries already done.")
print("-" * 70)


def _append_csv(path, rows):
    """Append a list-of-dicts to a CSV, writing header only on first creation."""
    if not rows:
        return
    df      = pd.DataFrame(rows)
    write_h = not path.exists()
    df.to_csv(path, mode="a", index=False, header=write_h)


# ── In-memory accumulators (for display at the end only) ──────────────────
all_rows        = []
per_block_rows  = []
summary_rows    = []

print(f"GENERATE_PKL_FROM_BINARY = {GENERATE_PKL_FROM_BINARY}")
print("-" * 70)

for parser_name, meta in PARSER_METADATA.items():
    total_blocks = int(meta["total_blocks"])
    mrd          = float(meta.get("mrd", float("nan")))

    print(f"\nParser: {parser_name}  |  total_blocks={total_blocks}  |  MRD={mrd}")

    # Skip entire parser if summary already done
    if (parser_name,) in done_sum:
        print(f"  [RESUME] Summary already exists — skipping parser.")
        continue

    parser_pkl_dir = PKL_DIR / parser_name

    if GENERATE_PKL_FROM_BINARY:
        binary_parser_dir = BINARY_DIR / parser_name
        
        if not binary_parser_dir.is_dir():
            print(f"  [SKIP] Binary folder not found: {binary_parser_dir}")
            continue

        run_dirs = sorted(
            r for r in binary_parser_dir.iterdir()
            if r.is_dir() and r.name.startswith("run_")
        )
        if not run_dirs:
            print(f"  [SKIP] No run_* folders found in {binary_parser_dir}")
            continue

        print(f"  Runs found (binary): {len(run_dirs)}")
        run_pkls = []

        for run_dir in run_dirs:
            bin_p = run_dir / "ft_input_cov.bin"
            idx_p = run_dir / "ft_input_cov.idx"
            if not bin_p.exists() or not idx_p.exists():
                print(f"    [SKIP] {run_dir.name}: missing .bin or .idx")
                continue
            pkl_name = f"{parser_name}_{run_dir.name}.pkl"
            pkl_path = parser_pkl_dir / pkl_name
            print(f"    [GEN]  {run_dir.name} -> {pkl_path} ...", end=" ", flush=True)
            try:
                stats = generate_pkl_from_binary(bin_p, idx_p, pkl_path, total_blocks)
                print(f"ok  ({stats['shape'][0]:,} execs, {stats['file_size_mb']:.2f} MB)")
                run_pkls.append((run_dir.name, pkl_path))
            except Exception as e:
                print(f"FAILED: {e}")

    else:
        print(f"  Binary parser folder: {parser_pkl_dir}")
        if not parser_pkl_dir.is_dir():
            print(f"  [SKIP] PKL folder not found: {parser_pkl_dir}")
            continue
        pkl_files = sorted(parser_pkl_dir.glob(f"{parser_name}_run_*.pkl"))
        if not pkl_files:
            pkl_files = sorted(parser_pkl_dir.glob("*.pkl"))
        if not pkl_files:
            print(f"  [SKIP] No pkl files found in {parser_pkl_dir}")
            continue
        run_pkls = [(p.stem, p) for p in pkl_files]
        print(f"  Runs found (pkl): {len(run_pkls)}")

    # ── Load iCFG and per-block ground truth ──────────────────────────────
    try:
        icfg         = load_icfg(parser_name)
        ground_truth = load_ground_truth(parser_name)
    except FileNotFoundError as e:
        print(f"  [SKIP] Missing iCFG or ground truth: {e}")
        continue

    all_block_ids = set(int(k) for k in icfg.keys())

    skipped_runs = []
    gotu_ests_final = []
    coverage_pcts_final, missed_pcts_final         = [], []

    for run_name, pkl_path in run_pkls:
        try:
            matrix = load_pkl(pkl_path)
            n      = matrix["shape"][0]
        except Exception as e:
            print(f"    [SKIP] {run_name}: {type(e).__name__}: {e}")
            skipped_runs.append(run_name)
            continue

        col_idx = matrix["col_idx"]
        row_ptr = matrix["row_ptr"]

        for ckpt in CHECKPOINTS:
            key = (parser_name, run_name, ckpt)

            # ── Skip if already written ────────────────────────────
            if key in done_ckpt:
                print(f"    [RESUME] {run_name} t={ckpt:.0%} already done — skipping.")
                # Still need scalars for end-of-campaign summary
                if ckpt == 1.00 and CKPT_CSV.exists():
                    try:
                        _row = pd.read_csv(CKPT_CSV).query(
                            "parser==@parser_name and run==@run_name and checkpoint==@ckpt"
                        ).iloc[0]
                        gotu_ests_final.append(_row["gotu_est"])
                        coverage_pcts_final.append(_row["blocks_seen"] / len(all_block_ids) * 100)
                        missed_pcts_final.append(_row["rho"] * 100)
                    except Exception:
                        pass
                continue

            S_t, n_t, block_counts = build_prefix(matrix, ckpt)
            frontier               = compute_frontier(S_t, icfg)

            # ── GoTu ──────────────────────────────────────────────
            v1       = sum(1 for c in block_counts.values() if c == 1)
            gotu_est = v1 / n_t if n_t > 0 else 0.0



            # ── Saturation rho(t) ─────────────────────────────────
            blocks_seen = len(S_t)
            rho_t       = (len(all_block_ids) - blocks_seen) / len(all_block_ids)

            # ── Hy-path estimates ─────────────────────────────────
            hy_estimates, hy_methods = compute_all_hy_path_estimates(
                S_t, frontier, icfg, block_counts, n_t, all_block_ids
            )

            # ── Per-block log error ───────────────────────────────
            per_block  = compute_per_block_log_error(hy_estimates, ground_truth, icfg, hy_methods)
            depth_summ = summarise_log_error_by_depth(per_block)

            print(f"    {run_name} t={ckpt:.0%}: n_t={n_t}  "
                  f"GoTu={gotu_est:.4f}  rho={rho_t:.4f}  "
                  f"|frontier|={len(frontier)}")

            # ── Build checkpoint row ──────────────────────────────
            row_dict = {
                "parser":      parser_name,
                "run":         run_name,
                "mrd":         mrd,
                "checkpoint":  ckpt,
                "n_t":         n_t,
                "n_frontier":  len(frontier),
                "gotu_est":    gotu_est,
                "rho":         rho_t,
                "blocks_seen": blocks_seen,
            }
            for bin_name in ["shallow", "medium", "deep"]:
                stats = depth_summ.get(bin_name, {})
                row_dict[f"log_err_{bin_name}"]  = stats.get("mean_log_err", float("nan"))
                row_dict[f"zc_rate_{bin_name}"]  = stats.get("zc_rate",      float("nan"))
                row_dict[f"n_blocks_{bin_name}"] = stats.get("n_blocks",     0)

            # ── Append to CSVs immediately ────────────────────────
            _append_csv(CKPT_CSV, [row_dict])

            pb_rows_out = []
            for pb in per_block:
                pb.update({"parser": parser_name, "run": run_name,
                            "checkpoint": ckpt, "mrd": mrd})
                pb_rows_out.append(pb)
            _append_csv(PB_CSV, pb_rows_out)

            # ── Accumulate for in-memory display ──────────────────
            all_rows.append(row_dict)
            per_block_rows.extend(pb_rows_out)

            # ── Mark as done so a re-run in same session skips it ─
            done_ckpt.add(key)

            # ── End-of-campaign scalars ───────────────────────────
            if ckpt == 1.00:
                gotu_ests_final.append(gotu_est)
                coverage_pcts_final.append(blocks_seen / len(all_block_ids) * 100)
                missed_pcts_final.append(rho_t * 100)

    if skipped_runs:
        print(f"  [WARNING] {len(skipped_runs)} run(s) skipped: {skipped_runs}")
    if not gotu_ests_final:
        print(f"  [SKIP] {parser_name}: no valid runs — skipping summary")
        continue

    sum_row = {
        "parser":            parser_name,
        "mrd":               mrd,
        "total_blocks":      total_blocks,
        "n_runs":            len(run_pkls),
        "gotu_est_mean":     _mean(gotu_ests_final),
        "coverage_pct_mean": _mean(coverage_pcts_final),
        "coverage_pct_std":  _std(coverage_pcts_final),
        "missed_pct_mean":   _mean(missed_pcts_final),
        "missed_pct_std":    _std(missed_pcts_final),
    }
    _append_csv(SUM_CSV, [sum_row])
    summary_rows.append(sum_row)

    print(f"  => Coverage(t=100%)={_mean(coverage_pcts_final):.1f}%")
    print("-" * 70)

print(f"\nPipeline complete. {len(summary_rows)} new parser(s) processed.")


Resume state: 290 checkpoint rows already done.
              290   per-block rows already done.
              6 parser summaries already done.
----------------------------------------------------------------------
GENERATE_PKL_FROM_BINARY = False
----------------------------------------------------------------------

Parser: parsera  |  total_blocks=16084  |  MRD=35.82
  [RESUME] Summary already exists — skipping parser.

Parser: parserb  |  total_blocks=16084  |  MRD=35.82
  [RESUME] Summary already exists — skipping parser.

Parser: parserc  |  total_blocks=10484  |  MRD=24.81
  [RESUME] Summary already exists — skipping parser.

Parser: parserd  |  total_blocks=17684  |  MRD=38.64
  [RESUME] Summary already exists — skipping parser.

Parser: parsere  |  total_blocks=26684  |  MRD=53.75
  [RESUME] Summary already exists — skipping parser.

Parser: parserf  |  total_blocks=9540  |  MRD=75.93
  [RESUME] Summary already exists — skipping parser.

Pipeline complete. 0 new parser(s) proc

---
## Section 9 — Save & Display Results

In [ ]:

if not CKPT_CSV.exists():
    print("No results found. Run the pipeline loop above first.")
else:
    ckpt_df = pd.read_csv(CKPT_CSV)
    print(f"Loaded: {CKPT_CSV}  ({len(ckpt_df)} rows)")

    ckpt_summary = (
        ckpt_df
        .groupby(["parser", "mrd", "checkpoint"])
        .agg(
            n_runs          = ("run",             "nunique"),
            gotu_est_mean   = ("gotu_est",         "mean"),
            rho_mean        = ("rho",               "mean"),
            log_err_shallow = ("log_err_shallow",   "mean"),
            log_err_medium  = ("log_err_medium",    "mean"),
            log_err_deep    = ("log_err_deep",      "mean"),
            zc_rate_shallow = ("zc_rate_shallow",   "mean"),
            zc_rate_medium  = ("zc_rate_medium",    "mean"),
            zc_rate_deep    = ("zc_rate_deep",      "mean"),
        )
        .reset_index()
    )
    ckpt_summary.to_csv(RESULTS_DIR / "summary_by_checkpoint.csv", index=False)
    print(f"Saved: summary_by_checkpoint.csv")

    if PB_CSV.exists():
        print(f"Exists: {PB_CSV}  ({sum(1 for _ in open(PB_CSV))-1} rows)")

    if SUM_CSV.exists():
        summary_df = pd.read_csv(SUM_CSV)
        print(f"Exists: {SUM_CSV}  ({len(summary_df)} rows)")


Loaded: results/per_run_per_checkpoint.csv  (10 rows)
Saved: summary_by_checkpoint.csv
Exists: results/summary.csv  (1 rows)


In [32]:
# Summary by checkpoint — log error across depth bins
if 'ckpt_summary' not in dir() or ckpt_summary is None:
    print("Run the Save cell above first.")
else:
    pd.set_option("display.float_format", "{:.5f}".format)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 220)

    print("\n=== CHECKPOINT SUMMARY — LOG ERROR BY DEPTH BIN ===")
    display(
        ckpt_summary[[
            "parser", "mrd", "checkpoint",
            "rho_mean", "gotu_est_mean",
            "log_err_shallow", "log_err_medium", "log_err_deep",
            "zc_rate_shallow", "zc_rate_medium", "zc_rate_deep",
        ]].sort_values(["parser", "checkpoint"]).reset_index(drop=True)
    )

    if 'summary_df' in dir() and summary_df is not None:
        print("\n=== END-OF-CAMPAIGN SUMMARY (t=100%) ===")
        display(
            summary_df[[
                "parser", "mrd", "gotu_est_mean",
                "coverage_pct_mean", "missed_pct_mean",
            ]].sort_values("mrd").reset_index(drop=True)
        )



=== CHECKPOINT SUMMARY — LOG ERROR BY DEPTH BIN ===


,parser,mrd,checkpoint,rho_mean,gotu_est_mean,log_err_shallow,log_err_medium,log_err_deep,zc_rate_shallow,zc_rate_medium,zc_rate_deep
0,parsera,35.82000,0.10000,0.92085,0.00002,NaN,NaN,NaN,NaN,NaN,NaN
1,parsera,35.82000,0.25000,0.91644,0.00001,NaN,NaN,NaN,NaN,NaN,NaN
2,parsera,35.82000,0.50000,0.91289,0.00000,NaN,NaN,NaN,NaN,NaN,NaN
3,parsera,35.82000,0.75000,0.91128,0.00000,NaN,NaN,NaN,NaN,NaN,NaN
4,parsera,35.82000,1.00000,0.90916,0.00000,NaN,NaN,NaN,NaN,NaN,NaN



=== END-OF-CAMPAIGN SUMMARY (t=100%) ===


,parser,mrd,gotu_est_mean,coverage_pct_mean,missed_pct_mean
0,parsera,35.82000,0.00000,9.08356,90.91644


---
## Section 11 — Convergence Analysis (Figure 3)

**Plot 1 :** For each parser, the hardest-to-reach block
(lowest true Pr(b) that is reached in at least one run) is selected as the target.
We split sample size from 0.1% to 10% of total executions and compute log-err for
estimators: Good-Turing, and Structure-aware (Struct).

**Plot 2 ( depth stratification):** log-err is
averaged over *all* unreached blocks within each depth bin (shallow/medium/deep),
showing how estimator accuracy degrades with decision depth.

In [ ]:
# ── Section 11: Convergence Analysis ─────────
# ── Set to True to load from existing CSVs; False to re-read from pipeline output ──
LOAD_FROM_CSV = True   

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

CONV_RESULTS_DIR = RESULTS_DIR / "convergence"
CONV_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Load CSVs ──────────────────────────────────────────────────────────────
if LOAD_FROM_CSV:
    if 'ckpt_df' not in dir() or ckpt_df is None:
        if CKPT_CSV.exists():
            ckpt_df = pd.read_csv(CKPT_CSV)
            print(f"Loaded per_run_per_checkpoint.csv: {len(ckpt_df)} rows")
        else:
            print("ERROR: per_run_per_checkpoint.csv not found. Run the pipeline cell first.")
            ckpt_df = None
    else:
        print(f"Using in-memory ckpt_df: {len(ckpt_df)} rows")

    if 'pb_df' not in dir() or pb_df is None:
        if PB_CSV.exists():
            pb_df = pd.read_csv(PB_CSV)
            print(f"Loaded per_block_log_error.csv: {len(pb_df)} rows")
        else:
            print("WARNING: per_block_log_error.csv not found. Plot 2 will use fallback.")
            pb_df = None
    else:
        print(f"Using in-memory pb_df: {len(pb_df)} rows")
else:
    # Force fresh read from disk
    if not CKPT_CSV.exists():
        print("ERROR: per_run_per_checkpoint.csv not found. Run the pipeline cell first.")
        ckpt_df = None
    else:
        ckpt_df = pd.read_csv(CKPT_CSV)
        print(f"Re-loaded per_run_per_checkpoint.csv: {len(ckpt_df)} rows")

    if not PB_CSV.exists():
        print("WARNING: per_block_log_error.csv not found. Plot 2 will use fallback.")
        pb_df = None
    else:
        pb_df = pd.read_csv(PB_CSV)
        print(f"Re-loaded per_block_log_error.csv: {len(pb_df)} rows")

# ══════════════════════════════════════════════════════════════════════════
# Plot 1 — Estimator convergence vs checkpoint 
# X-axis: checkpoint fraction (5 points: 10%, 25%, 50%, 75%, 100%)
# Y-axis: mean log-err of GoTu estimator per parser
# ══════════════════════════════════════════════════════════════════════════
if 'ckpt_df' in dir() and ckpt_df is not None:
    parsers    = sorted(ckpt_df['parser'].unique())
    n_parsers  = len(parsers)
    colors     = plt.cm.tab10(np.linspace(0, 1, n_parsers))

    fig1, ax1 = plt.subplots(figsize=(9, 5))

    plot1_rows = []
    for parser_name, color in zip(parsers, colors):
        pdf = ckpt_df[ckpt_df['parser'] == parser_name].copy()
        pdf = pdf.groupby('checkpoint').agg(
            gotu_mean  = ('gotu_est', 'mean'),
            rho_mean   = ('rho',      'mean'),
        ).reset_index().sort_values('checkpoint')

        # GoTu log-err: |log10(gotu_est) - log10(rho)|
        # rho is the saturation ground truth (fraction of blocks not yet seen)
        valid = pdf[(pdf['gotu_mean'] > 0) & (pdf['rho_mean'] > 0)]
        if valid.empty:
            continue
        log_err = np.abs(
            np.log10(valid['gotu_mean'].values) -
            np.log10(valid['rho_mean'].values)
        )
        mrd = ckpt_df[ckpt_df['parser'] == parser_name]['mrd'].iloc[0]
        ax1.plot(valid['checkpoint'].values * 100, log_err,
                 marker='o', color=color, label=f"{parser_name} (MRD={mrd})", lw=1.8)

        # Accumulate data for CSV
        for ckpt_val, gotu_m, rho_m, le in zip(
            valid['checkpoint'].values,
            valid['gotu_mean'].values,
            valid['rho_mean'].values,
            log_err,
        ):
            plot1_rows.append({
                'parser':         parser_name,
                'mrd':            mrd,
                'checkpoint':     ckpt_val,
                'checkpoint_pct': ckpt_val * 100,
                'gotu_mean':      gotu_m,
                'rho_mean':       rho_m,
                'log_err_gotu':   le,
            })

    # Save Plot 1 data CSV
    if plot1_rows:
        plot1_csv = CONV_RESULTS_DIR / "convergence_gotu_per_parser.csv"
        pd.DataFrame(plot1_rows).to_csv(plot1_csv, index=False)
        print(f"Saved CSV: {plot1_csv}")

    ax1.axhline(1.0, color='gray', ls='--', lw=0.8, alpha=0.7, label='log-err=1')
    ax1.set_xlabel("Checkpoint (% of total executions)")
    ax1.set_ylabel("log-err  |log\u2081\u2080(GoTu) \u2212 log\u2081\u2080(\u03c1)|")
    ax1.set_title("Plot 1: GoTu Estimator Convergence vs Checkpoint\n"
                  "(\u03c1 = saturation ground truth)")
    ax1.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    ax1.set_ylim(bottom=0)
    fig1.tight_layout()
    plot1_path = CONV_RESULTS_DIR / "convergence_gotu_per_parser.png"
    fig1.savefig(plot1_path, dpi=150, bbox_inches='tight')
    plt.close(fig1)
    print(f"Saved: {plot1_path}")

# ══════════════════════════════════════════════════════════════════════════
# Plot 2 — Depth-stratified log-err vs checkpoint
# Uses log_err_shallow / log_err_medium / log_err_deep

# ══════════════════════════════════════════════════════════════════════════
if 'ckpt_df' in dir() and ckpt_df is not None:
    depth_bins  = ['shallow', 'medium', 'deep']
    bin_colors  = {'shallow': 'tab:blue', 'medium': 'tab:orange', 'deep': 'tab:green'}
    parsers     = sorted(ckpt_df['parser'].unique())

   
    if pb_df is not None:
        
        pb_agg = (
            pb_df[pb_df['log_err'].notna()]
            .groupby(['checkpoint', 'depth_bin'])['log_err']
            .mean()
            .reset_index()
        )

        # Save Plot 2 data CSV (from per_block_log_error)
        plot2_csv = CONV_RESULTS_DIR / "convergence_by_depth.csv"
        pb_agg_out = pb_agg.copy()
        pb_agg_out['checkpoint_pct'] = pb_agg_out['checkpoint'] * 100
        pb_agg_out = pb_agg_out.rename(columns={'log_err': 'mean_log_err'})
        pb_agg_out[['checkpoint', 'checkpoint_pct', 'depth_bin', 'mean_log_err']].to_csv(plot2_csv, index=False)
        print(f"Saved CSV: {plot2_csv}")

        fig2, ax2 = plt.subplots(figsize=(9, 5))
        for bin_name in depth_bins:
            bdf = pb_agg[pb_agg['depth_bin'] == bin_name].sort_values('checkpoint')
            if bdf.empty:
                continue
            ax2.plot(bdf['checkpoint'].values * 100, bdf['log_err'].values,
                     marker='o', color=bin_colors[bin_name],
                     label=f"{bin_name}", lw=2)
        ax2.axhline(1.0, color='gray', ls='--', lw=0.8, alpha=0.7)
        ax2.set_xlabel("Checkpoint (% of total executions)")
        ax2.set_ylabel("Mean log-err  (Struct estimator)")
        ax2.set_title("Plot 2: Depth-stratified Log-error vs Checkpoint\n"
                      "(Structure-aware estimator; mean over all parsers & runs)")
        ax2.legend(fontsize=10)
        ax2.set_ylim(bottom=0)
        fig2.tight_layout()
        plot2_path = CONV_RESULTS_DIR / "convergence_by_depth.png"
        fig2.savefig(plot2_path, dpi=150, bbox_inches='tight')
        plt.close(fig2)
        print(f"Saved: {plot2_path}")

    else:
        
        ckpt_agg = (
            ckpt_df.groupby('checkpoint')
            .agg(
                log_err_shallow=('log_err_shallow', 'mean'),
                log_err_medium =('log_err_medium',  'mean'),
                log_err_deep   =('log_err_deep',    'mean'),
            )
            .reset_index()
            .sort_values('checkpoint')
        )

        # Save Plot 2 data CSV 
        plot2_rows = []
        for _, row in ckpt_agg.iterrows():
            for bin_name in depth_bins:
                val = row[f'log_err_{bin_name}']
                if not np.isnan(val):
                    plot2_rows.append({
                        'checkpoint':     row['checkpoint'],
                        'checkpoint_pct': row['checkpoint'] * 100,
                        'depth_bin':      bin_name,
                        'mean_log_err':   val,
                    })
        if plot2_rows:
            plot2_csv = CONV_RESULTS_DIR / "convergence_by_depth.csv"
            pd.DataFrame(plot2_rows).to_csv(plot2_csv, index=False)
            print(f"Saved CSV (fallback): {plot2_csv}")

        fig2, ax2 = plt.subplots(figsize=(9, 5))
        for bin_name in depth_bins:
            col = f'log_err_{bin_name}'
            valid = ckpt_agg[ckpt_agg[col].notna()]
            if valid.empty:
                continue
            ax2.plot(valid['checkpoint'].values * 100, valid[col].values,
                     marker='o', color=bin_colors[bin_name],
                     label=f"{bin_name}", lw=2)
        ax2.axhline(1.0, color='gray', ls='--', lw=0.8, alpha=0.7)
        ax2.set_xlabel("Checkpoint (% of total executions)")
        ax2.set_ylabel("Mean log-err  (Struct estimator)")
        ax2.set_title("Plot 2: Depth-stratified Log-error vs Checkpoint\n"
                      "(Structure-aware estimator; mean over all parsers & runs)")
        ax2.legend(fontsize=10)
        ax2.set_ylim(bottom=0)
        fig2.tight_layout()
        plot2_path = CONV_RESULTS_DIR / "convergence_by_depth.png"
        fig2.savefig(plot2_path, dpi=150, bbox_inches='tight')
        plt.close(fig2)
        print(f"Saved (fallback from ckpt CSV): {plot2_path}")

print("\nConvergence plots complete.")


Loaded per_run_per_checkpoint.csv: 290 rows
Loaded per_block_log_error.csv: 3411569 rows
Saved CSV: results/convergence/convergence_gotu_per_parser.csv
Saved: results/convergence/convergence_gotu_per_parser.png
Saved CSV: results/convergence/convergence_by_depth.csv
Saved: results/convergence/convergence_by_depth.png

Convergence plots complete.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Section 11b — Per-Parser Depth-stratified Convergence
# For each parser: mean log-err per depth bin across all runs, vs checkpoint.
# ══════════════════════════════════════════════════════════════════════════

PER_PARSER_DIR = CONV_RESULTS_DIR / "per_parser"
PER_PARSER_DIR.mkdir(parents=True, exist_ok=True)

depth_bins = ['shallow', 'medium', 'deep']
bin_colors = {'shallow': 'tab:blue', 'medium': 'tab:orange', 'deep': 'tab:green'}

# ── Ensure data is loaded ──────────────────────────────────────────────────
if 'ckpt_df' not in dir() or ckpt_df is None:
    if CKPT_CSV.exists():
        ckpt_df = pd.read_csv(CKPT_CSV)
        print(f"Loaded per_run_per_checkpoint.csv: {len(ckpt_df)} rows")
    else:
        print("ERROR: per_run_per_checkpoint.csv not found.")
        ckpt_df = None

if 'pb_df' not in dir() or pb_df is None:
    if PB_CSV.exists():
        pb_df = pd.read_csv(PB_CSV)
        print(f"Loaded per_block_log_error.csv: {len(pb_df)} rows")
    else:
        pb_df = None

if ckpt_df is None:
    print("No data available. Run the pipeline cell first.")
else:
    parsers = sorted(ckpt_df['parser'].unique())

    for parser_name in parsers:
        mrd = ckpt_df[ckpt_df['parser'] == parser_name]['mrd'].iloc[0]


        if pb_df is not None and 'parser' in pb_df.columns:
            parser_pb = pb_df[(pb_df['parser'] == parser_name) & pb_df['log_err'].notna()]
            if parser_pb.empty:
                print(f"  [{parser_name}] No per-block data — skipping.")
                continue
            agg = (
                parser_pb
                .groupby(['checkpoint', 'depth_bin'])['log_err']
                .mean()
                .reset_index()
                .rename(columns={'log_err': 'mean_log_err'})
            )
            agg['checkpoint_pct'] = agg['checkpoint'] * 100
            source_label = "per_block_log_error"
        else:
            parser_ckpt = ckpt_df[ckpt_df['parser'] == parser_name]
            agg_ckpt = (
                parser_ckpt.groupby('checkpoint')
                .agg(
                    log_err_shallow=('log_err_shallow', 'mean'),
                    log_err_medium =('log_err_medium',  'mean'),
                    log_err_deep   =('log_err_deep',    'mean'),
                )
                .reset_index()
                .sort_values('checkpoint')
            )
            rows = []
            for _, row in agg_ckpt.iterrows():
                for b in depth_bins:
                    val = row[f'log_err_{b}']
                    if not np.isnan(val):
                        rows.append({
                            'checkpoint':     row['checkpoint'],
                            'checkpoint_pct': row['checkpoint'] * 100,
                            'depth_bin':      b,
                            'mean_log_err':   val,
                        })
            agg = pd.DataFrame(rows)
            source_label = "ckpt_df_fallback"

        if agg.empty:
            print(f"  [{parser_name}] Empty aggregation — skipping.")
            continue

        # ── Save CSV ───────────────────────────────────────────────────────
        csv_path = PER_PARSER_DIR / f"convergence_by_depth_{parser_name}.csv"
        agg[['checkpoint', 'checkpoint_pct', 'depth_bin', 'mean_log_err']].to_csv(csv_path, index=False)
        print(f"Saved CSV: {csv_path}")

        # ── Plot ───────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(8, 4.5))
        for bin_name in depth_bins:
            bdf = agg[agg['depth_bin'] == bin_name].sort_values('checkpoint')
            if bdf.empty:
                continue
            ax.plot(bdf['checkpoint_pct'].values, bdf['mean_log_err'].values,
                    marker='o', color=bin_colors[bin_name],
                    label=bin_name, lw=2)

        ax.axhline(1.0, color='gray', ls='--', lw=0.8, alpha=0.7)
        ax.set_xlabel("Checkpoint (% of total executions)")
        ax.set_ylabel("Mean log-err  (Struct estimator)")
        ax.set_title(
            f"Depth-stratified Log-error vs Checkpoint\n"
            f"{parser_name}  (MRD={mrd})  —  mean over all runs"
        )
        ax.legend(fontsize=10)
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.3)
        fig.tight_layout()

        png_path = PER_PARSER_DIR / f"convergence_by_depth_{parser_name}.png"
        fig.savefig(png_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved PNG: {png_path}")

    print("\nPer-parser depth-stratified convergence plots complete.")


Saved CSV: results/convergence/per_parser/convergence_by_depth_parsera.csv
Saved PNG: results/convergence/per_parser/convergence_by_depth_parsera.png
Saved CSV: results/convergence/per_parser/convergence_by_depth_parserb.csv
Saved PNG: results/convergence/per_parser/convergence_by_depth_parserb.png
Saved CSV: results/convergence/per_parser/convergence_by_depth_parserc.csv
Saved PNG: results/convergence/per_parser/convergence_by_depth_parserc.png
Saved CSV: results/convergence/per_parser/convergence_by_depth_parserd.csv
Saved PNG: results/convergence/per_parser/convergence_by_depth_parserd.png
Saved CSV: results/convergence/per_parser/convergence_by_depth_parsere.csv
Saved PNG: results/convergence/per_parser/convergence_by_depth_parsere.png
Saved CSV: results/convergence/per_parser/convergence_by_depth_parserf.csv
Saved PNG: results/convergence/per_parser/convergence_by_depth_parserf.png

Per-parser depth-stratified convergence plots complete.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Section 11c — Greybox vs Blackbox Difference Plot 
# ══════════════════════════════════════════════════════════════════════════

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────
RESULTS_DIR = Path('results')
BB_CSV      = Path('results_all_blackbox') / 'structural' / 'per_run_per_checkpoint.csv'
GB_CSV      = Path('results') / 'structural' / 'per_run_per_checkpoint.csv'
OUT_DIR     = RESULTS_DIR / 'convergence' / 'bb_vs_gb'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEPTH_BINS   = ['shallow', 'medium', 'deep']
DEPTH_COLORS = {'shallow': '#1f77b4', 'medium': '#ff7f0e', 'deep': '#2ca02c'}
DEPTH_MARKS  = {'shallow': 'o', 'medium': 's', 'deep': '^'}

# ── Load and compute per-(parser, checkpoint, depth_bin) means ────────────
def compute_means(csv_path):
    raw = pd.read_csv(csv_path)
    rows = []
    for (parser, ckpt), grp in raw.groupby(['parser', 'checkpoint']):
        for bin_name in DEPTH_BINS:
            col = f'log_err_{bin_name}'
            if col in grp.columns:
                rows.append({'parser': parser,
                             'checkpoint': ckpt,
                             'depth_bin': bin_name,
                             'mean_log_err': grp[col].mean()})
    return pd.DataFrame(rows)

print(f"Loading BB: {BB_CSV}")
bb_means = compute_means(BB_CSV)
print(f"Loading GB: {GB_CSV}")
gb_means = compute_means(GB_CSV)

print("BB parsers:", sorted(bb_means['parser'].unique()))
print("GB parsers:", sorted(gb_means['parser'].unique()))

# ── Merge and compute difference ─────────────────────────────────────────
merged_all = pd.merge(
    bb_means.rename(columns={'mean_log_err': 'mean_log_err_bb'}),
    gb_means.rename(columns={'mean_log_err': 'mean_log_err_gb'}),
    on=['parser', 'checkpoint', 'depth_bin'],
    how='inner'
)
merged_all['diff'] = merged_all['mean_log_err_gb'] - merged_all['mean_log_err_bb']

all_parsers = sorted(merged_all['parser'].unique())
print("Parsers with both BB and GB:", all_parsers)

# ── Save per-parser CSVs ──────────────────────────────────────────────────
for parser, grp in merged_all.groupby('parser'):
    out_path = OUT_DIR / f'diff_{parser}.csv'
    grp.to_csv(out_path, index=False)
    print(f"  {parser}: mean diff by depth = "
          f"{grp.groupby('depth_bin')['diff'].mean().round(3).to_dict()}")

merged_all.to_csv(OUT_DIR / 'diff_all_parsers.csv', index=False)
print(f"\nSaved combined CSV → {OUT_DIR}/diff_all_parsers.csv")

# ── Plot ──────────────────────────────────────────────────────────────────
n = len(all_parsers)
ncols = 2
nrows = (n + 1) // 2
fig, axes = plt.subplots(nrows, ncols, figsize=(10, 4 * nrows), constrained_layout=True)
axes_flat = axes.flatten() if n > 1 else [axes]

for idx, parser in enumerate(all_parsers):
    ax = axes_flat[idx]
    data = merged_all[merged_all['parser'] == parser]

    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
    for bin_name in DEPTH_BINS:
        sub = data[data['depth_bin'] == bin_name].sort_values('checkpoint')
        ax.plot(sub['checkpoint'] * 100, sub['diff'],
                color=DEPTH_COLORS[bin_name],
                marker=DEPTH_MARKS[bin_name],
                markersize=4, linewidth=1.2, label=bin_name)
    ax.set_title(parser, fontsize=9, fontweight='bold')
    ax.set_xlabel('Checkpoint (%)', fontsize=8)
    ax.set_ylabel('Δ log-error (GB − BB)', fontsize=8)
    ax.set_xticks([10, 25, 50, 75, 100])
    ax.tick_params(labelsize=7)
    ax.grid(True, linewidth=0.3, alpha=0.5)

for ax in axes_flat[n:]:
    ax.set_visible(False)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=8,
           title='Depth bin', title_fontsize=8, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Greybox − Blackbox Log-Error Difference per Parser\n'
             '(positive = GB worse, negative = GB better)', fontsize=10)

out_png = OUT_DIR / 'diff_all_parsers.png'
fig.savefig(out_png, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f"Saved plot  → {out_png}")


Loading BB: results_all_blackbox/structural/per_run_per_checkpoint.csv
Loading GB: results/structural/per_run_per_checkpoint.csv
BB parsers: ['parsera', 'parserb', 'parserc', 'parserd', 'parsere', 'parserf']
GB parsers: ['parsera', 'parserb', 'parserc', 'parserd', 'parsere', 'parserf']
Parsers with both BB and GB: ['parsera', 'parserb', 'parserc', 'parserd', 'parsere', 'parserf']
  parsera: mean diff by depth = {'deep': -0.006, 'medium': -0.289, 'shallow': 0.088}
  parserb: mean diff by depth = {'deep': -0.021, 'medium': -0.3, 'shallow': 0.081}
  parserc: mean diff by depth = {'deep': -0.246, 'medium': 0.264, 'shallow': -0.074}
  parserd: mean diff by depth = {'deep': -0.456, 'medium': 0.228, 'shallow': -2.222}
  parsere: mean diff by depth = {'deep': -0.888, 'medium': 0.085, 'shallow': -2.04}
  parserf: mean diff by depth = {'deep': -1.077, 'medium': 0.384, 'shallow': -1.499}

Saved combined CSV → results/convergence/bb_vs_gb/diff_all_parsers.csv
Saved plot  → results/convergence/bb_v

---

# Section 12 — Paper Tables & Figures (Table I–V, Figures 1–7)


| Mode | CSV |
|------|-----|
| Blackbox | `results_all_blackbox_v1/structural/per_run_per_checkpoint.csv` |
| Greybox  | `results/structural/per_run_per_checkpoint.csv` |


All tables are written to `results/paper/tables/` and figures to
`results/paper/figures/`.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import Image, display

PAPER_OUT = Path('results') / 'paper'
(PAPER_OUT / 'tables').mkdir(parents=True, exist_ok=True)
(PAPER_OUT / 'figures').mkdir(parents=True, exist_ok=True)

# internal parser name -> paper label (paper order A..E); parserb excluded (dup of parsera)
PARSER_MAP = {
    'parsera': 'ParserA',
    'parserb': 'ParserB',
    'parserc': 'ParserC',
    'parserd': 'ParserD',
    'parsere': 'ParserE',
}
PAPER_PARSERS = list(PARSER_MAP.keys())   # canonical order
LABEL = PARSER_MAP

BB_SRC = Path('results_all_blackbox/structural/per_run_per_checkpoint.csv')
GB_SRC = Path('results_all_greybox/structural/per_run_per_checkpoint.csv')

DEPTHS       = ['shallow', 'medium', 'deep']
DEPTH_LABEL  = {'shallow': 'Shallow', 'medium': 'Medium', 'deep': 'Deep'}
DEPTH_COLORS = {'shallow': '#4e79a7', 'medium': '#f28e2b', 'deep': '#59a14f'}
DEPTH_MARK   = {'shallow': 'o', 'medium': 's', 'deep': '^'}
CKPTS        = [0.10, 0.25, 0.50, 0.75, 1.00]

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})

def _load(path):
    df = pd.read_csv(path)
    return df[df['parser'].isin(PAPER_PARSERS)].copy()

bb_ckpt = _load(BB_SRC)   # blackbox per-run-per-checkpoint
gb_ckpt = _load(GB_SRC)   # greybox  per-run-per-checkpoint

def depth_means(df):
    """Mean log-error over the 20 runs, per (parser, checkpoint), per depth stratum."""
    return (df.groupby(['parser', 'checkpoint'])
              [['log_err_shallow', 'log_err_medium', 'log_err_deep']]
              .mean().reset_index())

bb_dm = depth_means(bb_ckpt)   # blackbox depth means
gb_dm = depth_means(gb_ckpt)   # greybox  depth means

def show_fig(fig, path):
    """Save a figure and display the PNG inline (robust to any backend)."""
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    display(Image(str(path)))

print(f"Blackbox rows: {len(bb_ckpt)} | Greybox rows: {len(gb_ckpt)}")
print("Parser mapping:", {k: LABEL[k] for k in PAPER_PARSERS})
print("Runs/parser (BB):", bb_ckpt.groupby('parser')['run'].nunique().to_dict())
print("Runs/parser (GB):", gb_ckpt.groupby('parser')['run'].nunique().to_dict())

---
### Table I — Structural characteristics of the five synthetic parsers

In [ ]:
# ── TABLE I — structural characteristics ───────────────────────────────────
# |N|  = number of non-terminals            
# |P|  = number of production alternatives  
# dmax = maximum decision depth              
# #blocks = number of basic blocks           
GRAMMAR_FILE = {
    'parsera': 'parsers/parsera/grammar_mrd1.json',
    'parserb': 'parsers/parserb/grammar_mrd30.json',
    'parserc': 'parsers/parserc/grammar_mrd50.json',
    'parserd': 'parsers/parserd/grammar_mrd75.json',
    'parsere': 'parsers/parsere/grammar_mrd100.json',
}

rows = []
for p in PAPER_PARSERS:
    g    = json.load(open(GRAMMAR_FILE[p]))
    icfg = json.load(open(f'icfg/{p}.json'))
    rows.append({
        'Parser':  LABEL[p],
        '|N|':     len(g),
        '|P|':     sum(len(v) for v in g.values()),
        'dmax':    max(n['depth'] for n in icfg.values()),
        '#blocks': len(icfg),
    })
table1 = pd.DataFrame(rows)
table1.to_csv(PAPER_OUT / 'tables' / 'table1_structural.csv', index=False)

print("TABLE I — Structural characteristics of the five synthetic parsers")
print("|N|: non-terminals  |P|: production rules  dmax: max decision depth  #blocks: basic blocks")
display(table1)

---
### Table II — Structure-aware mean log-error, averaged over all parsers, per checkpoint (Blackbox)

In [ ]:
# ── TABLE II / III helper — mean log-error per depth x checkpoint, avg over parsers ──
def checkpoint_depth_table(dm):
    t = (dm.groupby('checkpoint')[['log_err_shallow', 'log_err_medium', 'log_err_deep']]
           .mean().reindex(CKPTS))
    t.index = [f'{int(c*100)}' for c in t.index]
    out = t.T.round(2)
    out.index = ['Shallow', 'Medium', 'Deep']
    out.index.name = 'Depth'
    out.columns.name = 'Checkpoint (%)'
    return out

table2 = checkpoint_depth_table(bb_dm)
table2.to_csv(PAPER_OUT / 'tables' / 'table2_struct_blackbox.csv')
print("TABLE II — Structure-aware mean log-error, averaged over all parsers (Blackbox). Threshold = 1.")
display(table2)

---
### Table III — Structure-aware mean log-error, averaged over all parsers, per checkpoint (Greybox)

In [ ]:
# ── TABLE III — greybox counterpart of Table II ────────────────────────────
table3 = checkpoint_depth_table(gb_dm)
table3.to_csv(PAPER_OUT / 'tables' / 'table3_struct_greybox.csv')
print("TABLE III — Structure-aware mean log-error, averaged over all parsers (Greybox). Threshold = 1.")
display(table3)

---
### Table IV — Final-checkpoint (100%) mean log-error, per parser and mode

In [ ]:
# ── TABLE IV — 100% checkpoint, per parser, per mode, per depth stratum ─────
rows = []
for p in PAPER_PARSERS:
    for mode, dm in [('Blackbox', bb_dm), ('Greybox', gb_dm)]:
        d = dm[(dm.parser == p) & (dm.checkpoint == 1.0)].iloc[0]
        rows.append({
            'Parser':  LABEL[p],
            'Mode':    mode,
            'Shallow': round(d.log_err_shallow, 2),
            'Medium':  round(d.log_err_medium, 2),
            'Deep':    round(d.log_err_deep, 2),
        })
table4 = pd.DataFrame(rows)
table4.to_csv(PAPER_OUT / 'tables' / 'table4_final_per_parser.csv', index=False)
print("TABLE IV — Final-checkpoint (100%) mean log-error. Shallow 0-10, Medium 11-40, Deep 41+. Threshold = 1.")
display(table4)

---
### Table V — Greybox − Blackbox log-error difference per parser (Δ = GB − BB) at 100%

In [ ]:
# ── TABLE V — Delta = GB - BB at the 100% checkpoint ───────────────────────
rows = []
for p in PAPER_PARSERS:
    b = bb_dm[(bb_dm.parser == p) & (bb_dm.checkpoint == 1.0)].iloc[0]
    g = gb_dm[(gb_dm.parser == p) & (gb_dm.checkpoint == 1.0)].iloc[0]
    rows.append({
        'Parser':    LABEL[p],
        'Δ Shallow': round(g.log_err_shallow - b.log_err_shallow, 2),
        'Δ Medium':  round(g.log_err_medium  - b.log_err_medium,  2),
        'Δ Deep':    round(g.log_err_deep    - b.log_err_deep,    2),
    })
table5 = pd.DataFrame(rows)
table5.to_csv(PAPER_OUT / 'tables' / 'table5_gb_minus_bb.csv', index=False)
print("TABLE V — Δ = GB − BB at 100%. Negative = greybox better (lower log-error).")
display(table5)

---
## Figures 1–7

In [ ]:
# ── FIGURE 1 — Depth-stratified log-error vs checkpoint, averaged over parsers (Blackbox) ──
def plot_avg_depth(dm, title, fname):
    t = (dm.groupby('checkpoint')[['log_err_shallow', 'log_err_medium', 'log_err_deep']]
           .mean().reindex(CKPTS))
    x = [c * 100 for c in CKPTS]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for d in DEPTHS:
        ax.plot(x, t[f'log_err_{d}'].values, marker=DEPTH_MARK[d], color=DEPTH_COLORS[d],
                lw=2, label=f'{DEPTH_LABEL[d]}')
    ax.axhline(1.0, color='gray', ls='--', lw=0.9, label='log-err = 1')
    ax.set_xlabel('Checkpoint (% of total executions)')
    ax.set_ylabel('Mean log-error')
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_ylim(bottom=0)
    ax.legend()
    fig.tight_layout()
    show_fig(fig, PAPER_OUT / 'figures' / fname)

plot_avg_depth(bb_dm,
    'Fig. 1 — Depth-stratified log-error vs checkpoint (Blackbox)',
    'fig1_depth_blackbox.png')

In [ ]:
# ── FIGURE 2 — Depth-stratified log-error per parser (Blackbox) ─────────────
def plot_per_parser_depth(dm, title, fname):
    fig, axes = plt.subplots(1, 5, figsize=(17, 3.4))
    x = [c * 100 for c in CKPTS]
    for ax, p in zip(axes, PAPER_PARSERS):
        sub = dm[dm.parser == p].set_index('checkpoint').reindex(CKPTS)
        for d in DEPTHS:
            ax.plot(x, sub[f'log_err_{d}'].values, marker=DEPTH_MARK[d],
                    color=DEPTH_COLORS[d], lw=1.6, ms=4, label=DEPTH_LABEL[d])
        ax.axhline(1.0, color='gray', ls='--', lw=0.8)
        ax.set_title(LABEL[p])
        ax.set_xlabel('Checkpoint (%)')
        ax.set_xticks(x)
        ax.set_ylim(bottom=0)
    axes[0].set_ylabel('Mean log-error')
    axes[0].legend(fontsize=8, loc='upper left')
    fig.suptitle(title, y=1.03)
    fig.tight_layout()
    show_fig(fig, PAPER_OUT / 'figures' / fname)

plot_per_parser_depth(bb_dm,
    'Fig. 2 — Depth-stratified log-error per parser (Blackbox)',
    'fig2_per_parser_blackbox.png')

In [ ]:
# ── FIGURES 3 & 4 — GoTu estimator convergence per parser ──────────────────
def plot_gotu(df, title, fname):
    x_all = [c * 100 for c in CKPTS]
    colors = plt.cm.tab10(np.linspace(0, 1, len(PAPER_PARSERS)))
    fig, ax = plt.subplots(figsize=(8, 5))
    for p, c in zip(PAPER_PARSERS, colors):
        g = (df[df.parser == p].groupby('checkpoint')
               .agg(gotu=('gotu_est', 'mean'), rho=('rho', 'mean')).reindex(CKPTS))
        m = (g.gotu > 0) & (g.rho > 0)
        if not m.any():
            continue
        le = np.abs(np.log10(g.gotu[m]) - np.log10(g.rho[m]))
        ax.plot(np.array(g.index[m]) * 100, le.values,
                marker='o', color=c, lw=1.8, label=LABEL[p])
    ax.axhline(1.0, color='gray', ls='--', lw=0.9, label='log-err = 1')
    ax.set_xlabel('Checkpoint (% of total executions)')
    ax.set_ylabel(r'log-error  $|\log_{10}(\mathrm{GoTu}) - \log_{10}(\rho)|$')
    ax.set_title(title)
    ax.set_xticks(x_all)
    ax.set_ylim(bottom=0)
    ax.legend()
    fig.tight_layout()
    show_fig(fig, PAPER_OUT / 'figures' / fname)

plot_gotu(bb_ckpt,
    'Fig. 3 — GoTu estimator convergence vs checkpoint (Blackbox)',
    'fig3_gotu_blackbox.png')
plot_gotu(gb_ckpt,
    'Fig. 4 — GoTu estimator convergence vs checkpoint (Greybox)',
    'fig4_gotu_greybox.png')

In [ ]:
# ── FIGURE 5 — Depth-stratified log-error per parser (Greybox) ──────────────
plot_per_parser_depth(gb_dm,
    'Fig. 5 — Depth-stratified log-error per parser (Greybox)',
    'fig5_per_parser_greybox.png')

In [ ]:
# ── FIGURE 6 — Depth-stratified log-error vs checkpoint, averaged over parsers (Greybox) ──
plot_avg_depth(gb_dm,
    'Fig. 6 — Depth-stratified log-error vs checkpoint (Greybox)',
    'fig6_depth_greybox.png')

In [ ]:
# ── FIGURE 7 — Per-parser Δ log-error (GB − BB) by depth stratum ────────────
# For each parser and checkpoint: diff = mean_log_err_GB - mean_log_err_BB, per depth.
merged = bb_dm.merge(gb_dm, on=['parser', 'checkpoint'], suffixes=('_bb', '_gb'))
for d in DEPTHS:
    merged[f'diff_{d}'] = merged[f'log_err_{d}_gb'] - merged[f'log_err_{d}_bb']
merged.to_csv(PAPER_OUT / 'tables' / 'fig7_gb_minus_bb_by_checkpoint.csv', index=False)

fig, axes = plt.subplots(1, 5, figsize=(17, 3.4))
x = [c * 100 for c in CKPTS]
for ax, p in zip(axes, PAPER_PARSERS):
    sub = merged[merged.parser == p].set_index('checkpoint').reindex(CKPTS)
    ax.axhline(0.0, color='gray', ls='--', lw=0.9)
    for d in DEPTHS:
        ax.plot(x, sub[f'diff_{d}'].values, marker=DEPTH_MARK[d],
                color=DEPTH_COLORS[d], lw=1.6, ms=4, label=DEPTH_LABEL[d])
    ax.set_title(LABEL[p])
    ax.set_xlabel('Checkpoint (%)')
    ax.set_xticks(x)
axes[0].set_ylabel('Δ log-error (GB − BB)')
axes[0].legend(fontsize=8, loc='best')
fig.suptitle('Fig. 7 — Per-parser Δ log-error (GB − BB). Negative = greybox better.', y=1.03)
fig.tight_layout()
show_fig(fig, PAPER_OUT / 'figures' / 'fig7_gb_minus_bb_per_parser.png')

print("All paper tables written to results/paper/tables/ and figures to results/paper/figures/.")